# 04 — Loop Agentico: Agente de Tesorería con Messages API

**Autor:** Gabriel Untiveros | **Fecha:** 2026-05-26

---

## Qué construimos aquí

El corazón del agente: el loop `while turns < max_turns` usando la SDK de Anthropic directamente.

**Sin frameworks. Sin LangChain. Solo la API.**

Adaptado de: `cwc-workshops/agent-decomposition/agents/before/stockpilot.py`

## Lo que el agente puede hacer

| Pregunta del usuario | Lo que ejecuta el agente |
|---|---|
| ¿Cómo estamos en caja? | `batch_dias_de_caja.py` → evalúa alertas |
| Genera el reporte semanal | `generar_reporte.py` → markdown completo |
| ¿Cuántos días de caja tiene CUENTA-002? | `rolling_mean_cashflow.py CUENTA-002 14` |
| ¿Qué es el nivel CRÍTICO? | `read_skill alerta-tesoreria/SKILL.md` |

## El patrón del loop (del workshop)

```python
while turns < max_turns:
    resp = client.messages.create(model=MODEL, tools=TOOLS, messages=messages)

    if resp.stop_reason == 'end_turn':   # agente terminó
        break

    for block in resp.content:            # ejecutar cada tool call
        if block.type == 'tool_use':
            result = dispatch(block.name, block.input)
            tool_results.append({...})

    messages.append({'role': 'assistant', 'content': resp.content})
    messages.append({'role': 'user',      'content': tool_results})
```

> Este patrón es **todo lo que necesitas** para el 80% de los agentes de producción.

In [3]:
import json, subprocess, sys, os
from pathlib import Path
from datetime import datetime

# Cargar API key desde .env si existe
env_file = Path('..') / '.env'
if env_file.exists():
    for line in env_file.read_text().splitlines():
        if '=' in line and not line.startswith('#'):
            k, v = line.split('=', 1)
            os.environ.setdefault(k.strip(), v.strip())

import anthropic

# Verificar API key
api_key = os.environ.get('ANTHROPIC_API_KEY', '')
assert api_key.startswith('sk-ant-'), (
    'ANTHROPIC_API_KEY no encontrada.\n'
    'Crea el archivo Skill_financiero/.env con:\n'
    'ANTHROPIC_API_KEY=sk-ant-...'
)

BASE        = Path('..').resolve()
SKILLS_BASE = BASE / '.claude' / 'skills'

# Verificar que los skills del NB 03 existen
required = [
    SKILLS_BASE / 'forecast-cashflow' / 'batch_dias_de_caja.py',
    SKILLS_BASE / 'alerta-tesoreria'  / 'SKILL.md',
    SKILLS_BASE / 'reporte-semanal'   / 'generar_reporte.py',
]
for r in required:
    assert r.exists(), f'Falta {r}. Ejecuta primero NB 03.'

MODEL = 'claude-sonnet-4-6'   # Sonnet: ideal para analisis financiero
client = anthropic.Anthropic(api_key=api_key)

print(f'Cliente Anthropic listo | Modelo: {MODEL}')
print(f'Skills disponibles: {[p.parent.name for p in SKILLS_BASE.glob("*/SKILL.md")]}')

Cliente Anthropic listo | Modelo: claude-sonnet-4-6
Skills disponibles: ['alerta-tesoreria', 'forecast-cashflow', 'reporte-semanal']


---
## Las 2 tools del agente

El agente tiene exactamente 2 herramientas. No más.

| Tool | Para qué | Por qué solo 2 |
|---|---|---|
| `bash_execute` | Ejecutar cualquier script Python de los skills | Un script procesa TODO el CSV en 1 call |
| `read_skill` | Leer un SKILL.md bajo demanda | Mantiene el prompt corto — la política entra solo cuando se necesita |

> Del workshop: la versión vieja de StockPilot tenía 12 tools (get_stock, get_sales_velocity, get_product, ...).  
> La versión nueva tiene 2. Menos tools = menos tokens = menos turns = agente más rápido y barato.

In [4]:
TOOL_DEFS = [
    {
        'name': 'bash_execute',
        'description': (
            'Ejecuta un script Python de los skills de tesoreria. '
            'Usa esto para: forecast de caja (batch_dias_de_caja.py), '
            'forecast individual (rolling_mean_cashflow.py CUENTA-ID 14), '
            'reporte semanal (generar_reporte.py). '
            'El comando debe ser una lista de strings como en subprocess.run().'
        ),
        'input_schema': {
            'type': 'object',
            'properties': {
                'command': {
                    'type': 'array',
                    'items': {'type': 'string'},
                    'description': 'Comando como lista. Ej: ["python", ".claude/skills/forecast-cashflow/batch_dias_de_caja.py"]'
                }
            },
            'required': ['command']
        }
    },
    {
        'name': 'read_skill',
        'description': (
            'Lee el contenido de un SKILL.md para cargar la politica bajo demanda. '
            'Skills disponibles: forecast-cashflow/SKILL.md, '
            'alerta-tesoreria/SKILL.md, reporte-semanal/SKILL.md.'
        ),
        'input_schema': {
            'type': 'object',
            'properties': {
                'skill_name': {
                    'type': 'string',
                    'description': 'Nombre del skill. Ej: alerta-tesoreria/SKILL.md'
                }
            },
            'required': ['skill_name']
        }
    }
]

print(f'Tools definidas: {[t["name"] for t in TOOL_DEFS]}')

Tools definidas: ['bash_execute', 'read_skill']


---
## El sistema prompt — corto por diseño

Del workshop: el prompt monolítico viejo tenía **402 líneas** hardcodeadas.  
El nuevo tiene **~15 líneas** y carga las políticas desde los SKILL.md cuando las necesita.

Beneficio directo: cada llamada a la API cuesta menos tokens porque el prompt es más corto.

In [5]:
SYSTEM_PROMPT = f'''
Eres un agente de tesoreria financiero. Analizas liquidez, calculas forecasts de caja,
evaluas alertas y generas reportes semanales.

HERRAMIENTAS:
- bash_execute: ejecuta scripts Python de los skills (ver lista abajo)
- read_skill: lee un SKILL.md para cargar la politica cuando la necesitas

SCRIPTS DISPONIBLES (usar con bash_execute):
  Todas las cuentas:  ["{sys.executable}", "{(SKILLS_BASE / 'forecast-cashflow' / 'batch_dias_de_caja.py').as_posix()}"]
  Cuenta individual:  ["{sys.executable}", "{(SKILLS_BASE / 'forecast-cashflow' / 'rolling_mean_cashflow.py').as_posix()}", "CUENTA-ID", "14"]
  Reporte semanal:    ["{sys.executable}", "{(SKILLS_BASE / 'reporte-semanal' / 'generar_reporte.py').as_posix()}"]

SKILLS (cargar con read_skill cuando la tarea lo requiera):
  forecast-cashflow/SKILL.md  -> cuando pidan forecast o proyeccion
  alerta-tesoreria/SKILL.md   -> cuando pidan alertas o revision de liquidez
  reporte-semanal/SKILL.md    -> cuando pidan el reporte semanal

REGLA PRINCIPAL: Usa UN script para procesar datos de todas las cuentas.
No hagas llamadas individuales por cuenta — ese es el anti-patron.
'''.strip()

print(f'System prompt: {len(SYSTEM_PROMPT.splitlines())} lineas | {len(SYSTEM_PROMPT)} chars')

System prompt: 19 lineas | 1265 chars


---
## Función dispatch — ejecuta cada tool call del agente

In [6]:
def dispatch(tool_name: str, tool_input: dict) -> str:
    """Ejecuta una tool y retorna el resultado como string."""

    if tool_name == 'bash_execute':
        command = tool_input['command']
        result  = subprocess.run(
            command,
            capture_output=True,
            text=True,
            cwd=str(BASE),           # directorio raiz del proyecto
            timeout=30
        )
        if result.returncode == 0:
            return result.stdout.strip()
        else:
            return f'ERROR (exit {result.returncode}): {result.stderr.strip()}'

    elif tool_name == 'read_skill':
        skill_path = SKILLS_BASE / tool_input['skill_name']
        if skill_path.exists():
            return skill_path.read_text(encoding='utf-8')
        else:
            return f'ERROR: Skill no encontrado: {skill_path}'

    return f'ERROR: Tool desconocida: {tool_name}'


# Test rapido del dispatch
test = dispatch('bash_execute', {'command': [sys.executable, str(SKILLS_BASE / 'forecast-cashflow' / 'rolling_mean_cashflow.py'), 'CUENTA-001', '14']})
print('Test dispatch bash_execute:')
print(test[:200], '...' if len(test) > 200 else '')
print()
test_skill = dispatch('read_skill', {'skill_name': 'alerta-tesoreria/SKILL.md'})
print(f'Test dispatch read_skill: {len(test_skill)} chars leidos')

Test dispatch bash_execute:
{"cuenta_id": "CUENTA-001", "moneda": "PEN", "saldo_actual": 356608.41, "forecast_saldo": 384173.11, "dias_de_caja": 999, "flujo_neto_dia": 1968.91, "horizon_dias": 14, "confidence": 0.85, "method": " ...

Test dispatch read_skill: 2326 chars leidos


---
## El loop agentico — adaptado de `stockpilot.py`

Este es el patrón central del workshop. Sin cambios conceptuales — solo el dominio cambia.

In [7]:
def run_agent(prompt: str, max_turns: int = 10, verbose: bool = True) -> dict:
    """
    Loop agentico base. Adaptado de:
    cwc-workshops/agent-decomposition/agents/before/stockpilot.py
    """
    messages   = [{'role': 'user', 'content': prompt}]
    tokens_in  = tokens_out = turns = 0
    final_text = ''
    tool_log   = []

    if verbose:
        print(f'USUARIO: {prompt}')
        print('-' * 60)

    while turns < max_turns:
        turns += 1

        resp = client.messages.create(
            model      = MODEL,
            max_tokens = 4096,
            system     = SYSTEM_PROMPT,
            tools      = TOOL_DEFS,
            messages   = messages,
        )

        tokens_in  += resp.usage.input_tokens
        tokens_out += resp.usage.output_tokens

        # El agente termino
        if resp.stop_reason == 'end_turn':
            final_text = ''.join(
                b.text for b in resp.content if b.type == 'text'
            )
            if verbose:
                print(f'AGENTE: {final_text}')
            break

        # Ejecutar tool calls
        tool_results = []
        for block in resp.content:
            if block.type == 'tool_use':
                if verbose:
                    print(f'  [Turn {turns}] Tool: {block.name} | Input: {json.dumps(block.input)[:80]}...')
                try:
                    result = dispatch(block.name, block.input)
                except Exception as e:
                    result = f'ERROR: {e}'

                tool_log.append({'turn': turns, 'tool': block.name, 'input': block.input})
                if verbose:
                    print(f'  [Turn {turns}] Resultado: {str(result)[:120]}...')

                tool_results.append({
                    'type'       : 'tool_result',
                    'tool_use_id': block.id,
                    'content'    : str(result),
                })

        if not tool_results:
            final_text = ''.join(
                b.text for b in resp.content if b.type == 'text'
            )
            break

        messages.append({'role': 'assistant', 'content': resp.content})
        messages.append({'role': 'user',      'content': tool_results})

    return {
        'final_text': final_text,
        'turns'     : turns,
        'tokens_in' : tokens_in,
        'tokens_out': tokens_out,
        'tool_log'  : tool_log,
    }

print('Funcion run_agent() lista.')

Funcion run_agent() lista.


---
## Test 1 — Consulta de liquidez

La pregunta más frecuente en tesorería. El agente debe:
1. Decidir usar `batch_dias_de_caja.py` (no llamar cuenta por cuenta)
2. Interpretar los resultados según el SKILL de alertas
3. Dar una respuesta accionable

In [8]:
resultado_1 = run_agent(
    '¿Cómo estamos en caja esta semana? Dame un resumen ejecutivo.',
    verbose=True
)
print()
print('=== MÉTRICAS ===')
print(f'  Turns:      {resultado_1["turns"]}')
print(f'  Tokens in:  {resultado_1["tokens_in"]:,}')
print(f'  Tokens out: {resultado_1["tokens_out"]:,}')
print(f'  Tools:      {[t["tool"] for t in resultado_1["tool_log"]]}')

USUARIO: ¿Cómo estamos en caja esta semana? Dame un resumen ejecutivo.
------------------------------------------------------------
  [Turn 1] Tool: read_skill | Input: {"skill_name": "alerta-tesoreria/SKILL.md"}...
  [Turn 1] Resultado: ---
name: alerta-tesoreria
description: >
  Reglas para determinar si una cuenta bancaria requiere accion urgente.
  Car...
  [Turn 1] Tool: read_skill | Input: {"skill_name": "reporte-semanal/SKILL.md"}...
  [Turn 1] Resultado: ---
name: reporte-semanal-tesoreria
description: >
  Estructura del reporte semanal de flujo de caja y alertas.
  Cargar...
  [Turn 1] Tool: bash_execute | Input: {"command": "[\"d:\\Proyecto_Gabriel\\.venv\\Scripts\\python.exe\", \"D:/Proyect...
  [Turn 1] Resultado: ERROR: [WinError 2] El sistema no puede encontrar el archivo especificado...
  [Turn 2] Tool: bash_execute | Input: {"command": ["d:\\Proyecto_Gabriel\\.venv\\Scripts\\python.exe", "D:\\Proyecto_G...
  [Turn 2] Resultado: # Reporte de Tesoreria â€” Semana del 26-M

---
## Test 2 — Reporte semanal completo

El agente debe ejecutar `generar_reporte.py` (un solo script = un solo tool call)
y presentar el resultado formateado. Verificar que **no** hace N tool calls individuales.

In [9]:
resultado_2 = run_agent(
    'Genera el reporte semanal de tesorería.',
    verbose=True
)
print()
print('=== MÉTRICAS ===')
print(f'  Turns:      {resultado_2["turns"]}')
print(f'  Tokens in:  {resultado_2["tokens_in"]:,}')
print(f'  Tokens out: {resultado_2["tokens_out"]:,}')
print(f'  Tools:      {[t["tool"] for t in resultado_2["tool_log"]]}')
print()
# Si el agente es eficiente: tool_log debe tener 1 sola entrada (generar_reporte.py)
n_tools = len(resultado_2['tool_log'])
print(f'Tool calls totales: {n_tools} {"OK (1 script = 1 call)" if n_tools == 1 else "(revisar — idealmente 1 call)"}')

USUARIO: Genera el reporte semanal de tesorería.
------------------------------------------------------------
  [Turn 1] Tool: read_skill | Input: {"skill_name": "reporte-semanal/SKILL.md"}...
  [Turn 1] Resultado: ---
name: reporte-semanal-tesoreria
description: >
  Estructura del reporte semanal de flujo de caja y alertas.
  Cargar...
  [Turn 1] Tool: bash_execute | Input: {"command": "[\"d:\\Proyecto_Gabriel\\.venv\\Scripts\\python.exe\", \"D:/Proyect...
  [Turn 1] Resultado: ERROR: [WinError 2] El sistema no puede encontrar el archivo especificado...
  [Turn 2] Tool: bash_execute | Input: {"command": ["python", "D:/Proyecto_Gabriel/Skill_financiero/.claude/skills/repo...
  [Turn 2] Resultado: # Reporte de Tesoreria â€” Semana del 26-May-2026

## Resumen Ejecutivo
- **Cuentas revisadas:** 3
- **Requieren atencio...
AGENTE: Aquí está el reporte semanal generado:

---

# 📊 Reporte de Tesorería — Semana del 26-May-2026

## Resumen Ejecutivo
- ✅ **3 cuentas revisadas** — ninguna requier

---
## Test 3 — Pregunta sobre una cuenta específica

El agente debe usar `rolling_mean_cashflow.py` con la cuenta específica
y luego cargar el SKILL de alertas para interpretar el resultado.

In [10]:
resultado_3 = run_agent(
    '¿Cuántos días de caja tiene CUENTA-002? ¿Debería preocuparme?',
    verbose=True
)
print()
print('=== MÉTRICAS ===')
print(f'  Turns:      {resultado_3["turns"]}')
print(f'  Tokens in:  {resultado_3["tokens_in"]:,}')
print(f'  Tokens out: {resultado_3["tokens_out"]:,}')
print(f'  Tools:      {[t["tool"] for t in resultado_3["tool_log"]]}')

USUARIO: ¿Cuántos días de caja tiene CUENTA-002? ¿Debería preocuparme?
------------------------------------------------------------
  [Turn 1] Tool: bash_execute | Input: {"command": "[\"d:\\Proyecto_Gabriel\\.venv\\Scripts\\python.exe\", \"D:/Proyect...
  [Turn 1] Resultado: ERROR: [WinError 2] El sistema no puede encontrar el archivo especificado...
  [Turn 1] Tool: read_skill | Input: {"skill_name": "alerta-tesoreria/SKILL.md"}...
  [Turn 1] Resultado: ---
name: alerta-tesoreria
description: >
  Reglas para determinar si una cuenta bancaria requiere accion urgente.
  Car...
  [Turn 2] Tool: bash_execute | Input: {"command": ["python", "D:/Proyecto_Gabriel/Skill_financiero/.claude/skills/fore...
  [Turn 2] Resultado: {"cuenta_id": "CUENTA-002", "moneda": "PEN", "saldo_actual": 224330.79, "forecast_saldo": 242378.9, "dias_de_caja": 999,...
AGENTE: ¡Datos obtenidos! Aquí está el análisis completo de **CUENTA-002**:

---

## 📊 Análisis de CUENTA-002

| Métrica | Valor |
|---|---|
| **Sa

---
## Comparativa de métricas entre los 3 tests

In [11]:
import pandas as pd

metricas = pd.DataFrame([
    {
        'Pregunta'   : '¿Cómo estamos en caja?',
        'Turns'      : resultado_1['turns'],
        'Tokens in'  : resultado_1['tokens_in'],
        'Tokens out' : resultado_1['tokens_out'],
        'Total tokens': resultado_1['tokens_in'] + resultado_1['tokens_out'],
        'Tool calls' : len(resultado_1['tool_log']),
    },
    {
        'Pregunta'   : 'Reporte semanal',
        'Turns'      : resultado_2['turns'],
        'Tokens in'  : resultado_2['tokens_in'],
        'Tokens out' : resultado_2['tokens_out'],
        'Total tokens': resultado_2['tokens_in'] + resultado_2['tokens_out'],
        'Tool calls' : len(resultado_2['tool_log']),
    },
    {
        'Pregunta'   : 'CUENTA-002 días de caja',
        'Turns'      : resultado_3['turns'],
        'Tokens in'  : resultado_3['tokens_in'],
        'Tokens out' : resultado_3['tokens_out'],
        'Total tokens': resultado_3['tokens_in'] + resultado_3['tokens_out'],
        'Tool calls' : len(resultado_3['tool_log']),
    },
])

print(metricas.to_string(index=False))
print()
costo_sonnet_input  = 3.00 / 1_000_000   # USD por token
costo_sonnet_output = 15.00 / 1_000_000
total_in  = metricas['Tokens in'].sum()
total_out = metricas['Tokens out'].sum()
costo = total_in * costo_sonnet_input + total_out * costo_sonnet_output
print(f'Costo estimado (3 queries): USD {costo:.4f}')
print(f'Costo por query: USD {costo/3:.4f}')

               Pregunta  Turns  Tokens in  Tokens out  Total tokens  Tool calls
 ¿Cómo estamos en caja?      3       8223        1010          9233           4
        Reporte semanal      3       6319         886          7205           3
CUENTA-002 días de caja      3       6288         853          7141           3

Costo estimado (3 queries): USD 0.1037
Costo por query: USD 0.0346


---
## Celda libre — prueba tus propias preguntas

Modifica `MI_PREGUNTA` y ejecuta esta celda para interactuar con el agente.

In [12]:
MI_PREGUNTA = '¿Cuál es la cuenta con más riesgo esta semana y qué harías al respecto?'

resultado_libre = run_agent(MI_PREGUNTA, verbose=True)
print(f'\nTokens totales: {resultado_libre["tokens_in"] + resultado_libre["tokens_out"]:,} | '
      f'Turns: {resultado_libre["turns"]} | '
      f'Tools: {[t["tool"] for t in resultado_libre["tool_log"]]}')

USUARIO: ¿Cuál es la cuenta con más riesgo esta semana y qué harías al respecto?
------------------------------------------------------------
  [Turn 1] Tool: read_skill | Input: {"skill_name": "alerta-tesoreria/SKILL.md"}...
  [Turn 1] Resultado: ---
name: alerta-tesoreria
description: >
  Reglas para determinar si una cuenta bancaria requiere accion urgente.
  Car...
  [Turn 1] Tool: bash_execute | Input: {"command": "[\"d:\\Proyecto_Gabriel\\.venv\\Scripts\\python.exe\", \"D:/Proyect...
  [Turn 1] Resultado: ERROR: [WinError 2] El sistema no puede encontrar el archivo especificado...
  [Turn 2] Tool: bash_execute | Input: {"command": ["d:/Proyecto_Gabriel/.venv/Scripts/python.exe", "D:/Proyecto_Gabrie...
  [Turn 2] Resultado: [
  {
    "cuenta_id": "CUENTA-001",
    "moneda": "PEN",
    "saldo_actual": 356608.41,
    "saldo_minimo": 50000,
    ...
AGENTE: Aquí está el análisis completo aplicando la política de alertas:

---

## 🔍 Análisis de Riesgo — Todas las Cuentas

| Cuenta | Mo

---
## Resumen NB 04 — Arquitectura completada

```
CAPA 1: SKILLS (bajo demanda)              ← NB 03
  forecast-cashflow/SKILL.md
  alerta-tesoreria/SKILL.md
  reporte-semanal/SKILL.md

CAPA 2: CODE EXECUTION (1 script = 1 call) ← NB 02 + NB 03
  rolling_mean_cashflow.py
  batch_dias_de_caja.py
  generar_reporte.py

CAPA 3: LOOP AGENTICO (Messages API)       ← NB 04
  run_agent(prompt) → respuesta en lenguaje natural
```

| Check | Estado |
|---|---|
| Loop `while turns < max_turns` funcional | OK |
| Tool `bash_execute` ejecuta scripts | OK |
| Tool `read_skill` carga SKILLs bajo demanda | OK |
| Agente responde preguntas de liquidez | OK |
| Agente genera reporte semanal | OK |
| Métricas de tokens y costo calculadas | OK |

**Siguiente fase (Semana 3-4):** `agents-that-remember/` — agregar Memory Store
para que el agente recuerde el contexto del cliente entre sesiones.

Este es el diferenciador comercial: pasar de 'goldfish' (olvida todo) a 'CFO interno'
(recuerda KPIs, decisiones pasadas y compromisos pendientes del cliente).